In [87]:
import rasterio
import xarray as xr
import glob
from pathlib import Path
import geopandas as gpd
from rasterio.mask import mask
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling
from scipy.ndimage import median_filter
from rasterio.io import MemoryFile
from rasterio.features import shapes
from shapely.geometry import shape
#from skimage.morphology import (
#    binary_opening, binary_closing, binary_erosion, binary_dilation,
#    disk, remove_small_objects, remove_small_holes
#)
#from scipy.ndimage import binary_fill_holes

import skimage as ski
import numpy as np

In [88]:
year = 2023

### 1) Define input and output folder

In [89]:
output_folder = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/openEO/")
output_folder.mkdir(exist_ok=True)

input_folder_s2 = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/openEO/S2/" + str(year))
input_folder_s1 = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/openEO/S1/" + str(year))

### 2) Find all files and load them

In [90]:
def find_files(input_folder, pattern):
    #Find files in the input folder matching the given pattern
    return sorted(glob.glob(str(input_folder / pattern)))

In [91]:
# find corresponding coherence file
#s1_coherence_files = find_files(input_folder_s1 , "sar_coherence_composite_all_swaths_thresh_0.3_masked.tiff")
#print(f"Found {len(s1_coherence_files)} S1 coherence files.")

# find acending and descending coherence files
s1_coherence_files_asc = find_files(input_folder_s1 , "threshold_ascending_*.tiff")
print(f"Found {len(s1_coherence_files_asc)} S1 coherence files (ascending).")

s1_coherence_files_desc = find_files(input_folder_s1 , "threshold_descending_*.tiff")
print(f"Found {len(s1_coherence_files_desc)} S1 coherence files (descending).")

# find corresponding S2 glacier outline file
s2_glacier_files_east = find_files(input_folder_s2 , "S2_glacier_east_*.tiff")
print(f"Found {len(s2_glacier_files_east)} S2 glacier outline files.")

s2_glacier_files_west = find_files(input_folder_s2 , "S2_glacier_west_*.tiff")
print(f"Found {len(s2_glacier_files_west)} S2 glacier outline files.")

# find corresponding NDSI file
#s2_ndsi_files = find_files(Path(input_folder_s2 / "ndsi" / str(year) ), "ndsi_composite_" + str(year) + "_0.4.tiff")
#print(f"Found {len(s2_ndsi_files)} S2 NDSI files.")

# find corresponding ratio file
#s2_ratio_files = find_files(Path(input_folder_s2 / "ratio" / str(year) ), "ratio_composite_" + str(year) + ".tiff")
#print(f"Found {len(s2_ratio_files)} S2 ratio files.")

with rasterio.open(s1_coherence_files_asc[0]) as src:
    print("Loading S1 coherence (ascending):", s1_coherence_files_asc)
    s1_asc = src.read()
    s1_asc_meta = src.meta
    
with rasterio.open(s1_coherence_files_desc[0]) as src:
    print("Loading S1 coherence (descending):", s1_coherence_files_desc)
    s1_desc = src.read()
    s1_desc_meta = src.meta

with rasterio.open(s2_glacier_files_east[0]) as src:
    print("Loading S2 glacier outline (east):", s2_glacier_files_east)  
    s2_east = src.read()
    s2_east_meta = src.meta

with rasterio.open(s2_glacier_files_west[0]) as src:
    print("Loading S2 glacier outline (west):", s2_glacier_files_west)  
    s2_west = src.read()
    s2_west_meta = src.meta
   

#with rasterio.open(s2_ndsi_files[0]) as src:
#    print("Loading S2 NDSI:", s2_ndsi_files)  
#    s2_ndsi = src.read()
#    s2_ndsi_meta = src.meta
    

#with rasterio.open(s2_ratio_files[0]) as src:
#    print("Loading S2 ratio:", s2_ratio_files)  
#    s2_ratio = src.read()
#    s2_ratio_meta = src.meta


Found 1 S1 coherence files (ascending).
Found 1 S1 coherence files (descending).
Found 1 S2 glacier outline files.
Found 1 S2 glacier outline files.
Loading S1 coherence (ascending): ['/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/openEO/S1/2023/threshold_ascending_0.4_2023.tiff']
Loading S1 coherence (descending): ['/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/openEO/S1/2023/threshold_descending_0.4_2023.tiff']
Loading S2 glacier outline (east): ['/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/openEO/S2/2023/S2_glacier_east_2023.tiff']
Loading S2 glacier outline (west): ['/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/openEO/S2/2023/S2_glacier_west_2023.tiff']


### 3) Merge East and West Sentinel-2 classification

In [92]:
with rasterio.open(s2_glacier_files_east[0]) as s2_east, \
     rasterio.open(s2_glacier_files_west[0]) as s2_west:

    mosaic, out_transform = merge(
        [s2_east, s2_west],
        method="max"
    )

    out_meta = s2_east.meta.copy()

    out_meta.update({
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_transform,
        "dtype": "uint8",
        "nodata": 0,
    })

    output_file = input_folder_s2 / f"glacier_composite_{year}_0.4.tiff"

    with rasterio.open(output_file, "w", **out_meta) as dest:
        dest.write(mosaic)

### 4) Align all rasters spatially

In [93]:
def align_to_reference(src_array, src_transform, src_crs, ref_meta, resampling=Resampling.nearest):
    #Resample src_array onto the reference grid defined by ref_meta.
    aligned = np.zeros((ref_meta['height'], ref_meta['width']), dtype=src_array.dtype)
    reproject(
        source=src_array,
        destination=aligned,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=ref_meta['transform'],
        dst_crs=ref_meta['crs'],
        resampling=resampling
    )
    return aligned

In [94]:
ref_meta = s1_asc_meta  

s2_aligned = align_to_reference(
    mosaic[0], out_transform, s2_east_meta['crs'], ref_meta, resampling=Resampling.nearest
)

s1_asc_aligned = align_to_reference(
    s1_asc, s1_asc_meta['transform'], s1_asc_meta['crs'], ref_meta, resampling=Resampling.nearest
)

s1_desc_aligned = align_to_reference(
    s1_desc, s1_desc_meta['transform'], s1_desc_meta['crs'], ref_meta, resampling=Resampling.nearest
)

### 5) Combine files and safe to disk

In [95]:
s2_mask = s2_aligned > 0

s1_asc_mask = np.nan_to_num(s1_asc_aligned, nan=0) > 0
s1_desc_mask = np.nan_to_num(s1_desc_aligned, nan=0) > 0

s1_mask = s1_asc_mask | s1_desc_mask

combined = (
    s2_mask |
    s1_asc_mask |
    s1_desc_mask
).astype(np.uint8)

out_meta = ref_meta.copy()

out_meta.update({
    "count": 1,
    "dtype": "uint8",
    "nodata": 0
})

combined_filename = (
    output_folder / "S1_S2" /
    f"s1_s2_asc_desc_combined_{year}_0.4.tiff"
)

with rasterio.open(combined_filename, "w", **out_meta) as dest:
    dest.write(combined, 1)


### 6) Clip to glacier outlines from 1997

In [ ]:
# mask the binary rasters with the glacier outlines shapefile
mask_filename = "/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/glacier_outline/glacier_inventory/SGIhom_1997_v2025.shp"

# Read glacier mask
glacier_mask = gpd.read_file(mask_filename)

# create function to mask the binary rasters with the glacier outlines shapefile
def mask_and_filter_raster(raster_path, mask_geometry, out_folder):

    with rasterio.open(raster_path) as src:

        # Reproject glacier polygons to raster CRS
        glacier_mask_reprojected = mask_geometry.to_crs(src.crs)

        # Mask/crop
        masked_array, masked_transform = mask(
            src,
            glacier_mask_reprojected.geometry,
            crop=True,
            filled=False
        )

        # Boolean mask: True = inside glacier
        valid_area = ~masked_array.mask[0]

        # Extract binary raster
        binary = masked_array[0].filled(0).astype(np.uint8)

        # Median filter = majority filter for binary 0/1 data
        filtered = median_filter(
            binary,
            size=3
        )

        # Make sure pixels outside glacier are 0
        filtered[~valid_area] = 0

        # Metadata
        masked_meta = src.meta.copy()

        masked_meta.update({
            "height": filtered.shape[0],
            "width": filtered.shape[1],
            "transform": masked_transform,
            "count": 1,
            "dtype": "uint8",
            "nodata": 0
        })

        masked_filename = (
            Path(out_folder) /
            f"{Path(raster_path).stem}_masked.tiff"
        )

        with rasterio.open(masked_filename, "w", **masked_meta) as dst:
            dst.write(filtered, 1)

    print("Saved:", masked_filename)

    return masked_filename


glac_masked = mask_and_filter_raster(
    combined_filename,
    glacier_mask,
    output_folder
)


### 7) Apply morphological filtering

In [96]:
def process_binary_mask(
    binary_mask,
    transform,
    crs,
    glacier_mask,
    disk_size=5
):
    """
    Mask a binary raster with glacier polygons and apply
    morphological opening.
    """

    # Convert boolean to uint8
    binary_mask = binary_mask.astype(np.uint8)

    # Create temporary raster in memory
    from rasterio.io import MemoryFile

    meta = {
        "driver": "GTiff",
        "height": binary_mask.shape[0],
        "width": binary_mask.shape[1],
        "count": 1,
        "dtype": "uint8",
        "crs": crs,
        "transform": transform,
        "nodata": 0,
    }

    with MemoryFile() as memfile:

        with memfile.open(**meta) as src:
            src.write(binary_mask, 1)

            glacier_mask_reprojected = glacier_mask.to_crs(crs)

            masked_array, masked_transform = mask(
                src,
                glacier_mask_reprojected.geometry,
                crop=True,
                filled=False
            )

    # Keep track of glacier area
    glacier_area = ~masked_array.mask[0]

    # Fill outside glacier with zero
    binary = masked_array[0].filled(0).astype(np.uint8)

    # Morphological opening
    struct = ski.morphology.disk(disk_size)

    opened = ski.morphology.opening(
        binary,
        struct
    ).astype(np.uint8)

    # apply morphological closing to fill small holes
    closed = ski.morphology.closing(
        opened,
        struct
    ).astype(np.uint8)

    # Force outside glacier to zero
    closed[~glacier_area] = 0

    return closed, masked_transform


In [97]:
s2_glacier, s2_transform = process_binary_mask(
    s2_mask,
    ref_meta["transform"],
    ref_meta["crs"],
    glacier_mask,
    disk_size=3
)

s1_glacier, s1_transform = process_binary_mask(
    s1_mask,
    ref_meta["transform"],
    ref_meta["crs"],
    glacier_mask,
    disk_size=3
)

combined_glacier, combined_transform = process_binary_mask(
    combined,
    ref_meta["transform"],
    ref_meta["crs"],
    glacier_mask,
    disk_size=3
)

 

### 8) Convert into polygons

In [98]:
def raster_to_polygons(binary_raster, transform, crs):

    results = [
        {
            "geometry": shape(geom),
            "value": int(value)
        }
        for geom, value in shapes(
            binary_raster.astype(np.uint8),
            transform=transform
        )
        if value == 1
    ]

    gdf = gpd.GeoDataFrame(
        results,
        crs=crs
    )

    return gdf

In [99]:
s2_gdf = raster_to_polygons(
    s2_glacier,
    s2_transform,
    ref_meta["crs"]
)

s1_gdf = raster_to_polygons(
    s1_glacier,
    s1_transform,
    ref_meta["crs"]
)

combined_gdf = raster_to_polygons(
    combined_glacier,
    combined_transform,
    ref_meta["crs"]
)


s2_path = (
    input_folder_s2 /
    f"s2_glacier_outlines_{year}_v0.4.shp"
)

s1_path = (
    input_folder_s1 /
    f"s1_glacier_outlines_{year}_v0.4.shp"
)

combined_path = (
    output_folder / "S1_S2" /
    f"s1_s2_glacier_outlines_{year}_v0.4.shp"
)

s2_gdf.to_file(s2_path)
s1_gdf.to_file(s1_path)
combined_gdf.to_file(combined_path)

print(f"S2: {len(s2_gdf)} polygons")
print(f"S1: {len(s1_gdf)} polygons")
print(f"S1+S2: {len(combined_gdf)} polygons")

S2: 258 polygons
S1: 378 polygons
S1+S2: 375 polygons


### 9) Extract area in km2

In [100]:
s2_area_km2 = s2_gdf.geometry.area.sum() / 1e6
s1_area_km2 = s1_gdf.geometry.area.sum() / 1e6
combined_area_km2 = combined_gdf.geometry.area.sum() / 1e6

print(f"S2 glacier area:       {s2_area_km2:.2f} km²")
print(f"S1 glacier area:       {s1_area_km2:.2f} km²")
print(f"S1 + S2 glacier area:  {combined_area_km2:.2f} km²")


S2 glacier area:       52.16 km²
S1 glacier area:       55.54 km²
S1 + S2 glacier area:  69.08 km²
